In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model)

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.6643837690353394
epoch:  1, loss: 0.36191779375076294
epoch:  2, loss: 0.20624038577079773
epoch:  3, loss: 0.12665294110774994
epoch:  4, loss: 0.08347078412771225
epoch:  5, loss: 0.06032742187380791
epoch:  6, loss: 0.048048168420791626
epoch:  7, loss: 0.041563525795936584
epoch:  8, loss: 0.03812287375330925
epoch:  9, loss: 0.0362933985888958
epoch:  10, loss: 0.035318244248628616
epoch:  11, loss: 0.03479543328285217
epoch:  12, loss: 0.03451119363307953
epoch:  13, loss: 0.03435305878520012
epoch:  14, loss: 0.034261081367731094
epoch:  15, loss: 0.03423897176980972
epoch:  16, loss: 0.03411790356040001
epoch:  17, loss: 0.03405411168932915
epoch:  18, loss: 0.03401844576001167
epoch:  19, loss: 0.03399692848324776
epoch:  20, loss: 0.03396910801529884
epoch:  21, loss: 0.03393556550145149
epoch:  22, loss: 0.033915985375642776
epoch:  23, loss: 0.03390209376811981
epoch:  24, loss: 0.03387231379747391
epoch:  25, loss: 0.033855587244033813
epoch:  26, loss: 

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7663923320261933
Test metrics:  R2 = 0.7198666929579376


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentTR(model=model, radius_init=1)

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.26045721769332886
epoch:  1, loss: 0.26045721769332886
epoch:  2, loss: 0.05912916362285614
epoch:  3, loss: 0.05912916362285614
epoch:  4, loss: 0.039901718497276306
epoch:  5, loss: 0.03414098173379898
epoch:  6, loss: 0.03414098173379898
epoch:  7, loss: 0.03414098173379898
epoch:  8, loss: 0.03408774361014366
epoch:  9, loss: 0.03407362848520279
epoch:  10, loss: 0.034060075879096985
epoch:  11, loss: 0.03404682129621506
epoch:  12, loss: 0.03403390571475029
epoch:  13, loss: 0.03402131423354149
epoch:  14, loss: 0.03400899097323418
epoch:  15, loss: 0.03399686887860298
epoch:  16, loss: 0.03398514166474342
epoch:  17, loss: 0.03397350013256073
epoch:  18, loss: 0.033961907029151917
epoch:  19, loss: 0.03395039215683937
epoch:  20, loss: 0.03393876180052757
epoch:  21, loss: 0.033927179872989655
epoch:  22, loss: 0.03391560539603233
epoch:  23, loss: 0.03390413522720337
epoch:  24, loss: 0.033892672508955
epoch:  25, loss: 0.03388119116425514
epoch:  26, loss: 0.

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.764012612277074
Test metrics:  R2 = 0.7147721970376322
